# Stage 1: Exploratory Data Analysis (EDA) - Scam Message Detection
This notebook explores the dataset properties, class distributions, message length distributions, and key vocabulary differences between legitimate (ham) and scam messages.

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.config import TRAIN_FILE, VAL_FILE, TEST_FILE
from src.preprocess import clean_text, extract_text_statistics
from src.data_loader import load_and_merge_datasets, create_stratified_splits

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load Processed Training Split
We inspect only the training data to avoid any exploratory data leakage.

In [ ]:
if not TRAIN_FILE.exists():
    print("Creating dataset splits...")
    create_stratified_splits()

df_train = pd.read_csv(TRAIN_FILE)
print(f"Train set shape: {df_train.shape}")
df_train.head()

## 2. Class Imbalance Analysis

In [ ]:
counts = df_train['label'].value_counts()
percentages = df_train['label'].value_counts(normalize=True) * 100
print("Class Counts:")
for label, name in [(0, 'Legitimate (Ham)'), (1, 'Scam/Spam')]:
    print(f"  {name} ({label}): {counts.get(label, 0)} ({percentages.get(label, 0):.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df_train, x='label', palette=['#22c55e', '#ef4444'], ax=ax)
ax.set_xticklabels(['Legitimate (0)', 'Scam (1)'])
ax.set_title('Class Distribution in Training Split')
plt.show()

## 3. Structural Characteristics: Message Length & Digit Density

In [ ]:
stats_list = [extract_text_statistics(t) for t in df_train['text']]
df_stats = pd.DataFrame(stats_list)
df_stats['label'] = df_train['label']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_stats, x='label', y='char_length', palette=['#22c55e', '#ef4444'], ax=axes[0])
axes[0].set_xticklabels(['Legitimate', 'Scam'])
axes[0].set_title('Character Length by Class')
axes[0].set_ylim(0, 300)

sns.boxplot(data=df_stats, x='label', y='digit_ratio', palette=['#22c55e', '#ef4444'], ax=axes[1])
axes[1].set_xticklabels(['Legitimate', 'Scam'])
axes[1].set_title('Digit Ratio (Numbers %)')
plt.tight_layout()
plt.show()

## 4. Top Vocabulary Terms by Class

In [ ]:
scam_words = " ".join(df_train[df_train['label'] == 1]['cleaned_text']).split()
legit_words = " ".join(df_train[df_train['label'] == 0]['cleaned_text']).split()

top_scam = Counter(scam_words).most_common(15)
top_legit = Counter(legit_words).most_common(15)

print("Top Tokens in Scam Messages:")
for word, count in top_scam:
    print(f"  {word:<15}: {count}")